> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# 金融データの活用 例 #1：アルトマンZスコア（ダブルプライム）の計算

アルトマン教授は、その名高い「Zスコア」を、NYU 在籍中の [1968年](http://www.defaultrisk.com/_pdf6j4/Financial_Ratios_Discriminant_Anlss_n_Prdctn_o_Crprt_Bnkrptc.pdf) に初めて定式化しました。「Zスコア」は、企業がデフォルトに陥る可能性を定量化しようとするものです。何度かの改良を経て、企業の信用リスクをより適切に定量化するために、アルトマンZスコア（ダブルプライム）が開発されました。アルトマン教授のプレゼンテーション（[こちら](http://pages.stern.nyu.edu/~ealtman/3-%20CopCrScoringModels.pdf)）では、このモデルを含むいくつかのモデルが順に解説されています。開発当初、スコアには大まかなカットオフ値があり、2.6 を超えれば企業は「健全」、1.1〜2.6 の間は「グレーゾーン」、1.1 未満では企業は倒産のリスクがあるとされていました。しかし、時間の経過とともに、その大まかな尺度は精緻化されていきました。今日のアルトマンZスコア（ダブルプライム）の独特な点の一つは、通常の信用格付けへの対応付けが存在することです。以下では、[Alpha Vantage のデータ](https://www.alphavantage.co/documentation/#fundamentals) を使って、スコアをゼロから計算する例を順に解説します（この例はもともと IEX Cloud API を使用していましたが、同 API は2024年8月に提供終了しました）。

$$ Z” = 6.56x_1 +3.26x_2 + 6.72x_3 + 1.05x_4 $$
$$ \textrm{ここで:} $$
$$ x_1 = \textrm{運転資本 / 総資産} $$
$$ x_2 = \textrm{利益剰余金 / 総資産} $$
$$ x_3 = \textrm{EBIT（利払前・税引前利益）/ 総資産} $$
$$ x_4 = \textrm{株式時価総額 / 負債総額} $$

In [ ]:
import os
import time
import requests

ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
# Alpha Vantage の公開 "demo" キーはそのまま動作しますが、実データが返るのは
# 以下で使用する "IBM" ティッカーのみであり、共有される demo トラフィックは
# 1秒あたり数リクエスト程度に制限されています。その他のティッカー（および、より高い
# 個人用のレート上限）については、https://www.alphavantage.co/support/#api-key で
# 無料キーを取得し、ここにハードコードするのではなく、環境変数として設定してください：
#   export ALPHA_VANTAGE_API_KEY=your_key_here
ALPHA_VANTAGE_API_KEY = os.environ.get("ALPHA_VANTAGE_API_KEY", "demo")

def av_fundamentals(function, ticker):
    '''Alpha Vantage からファンダメンタルズのエンドポイント（OVERVIEW、INCOME_STATEMENT、BALANCE_SHEET）を取得します。'''
    time.sleep(0.6)  # demo キーの共有レート制限を超えないようにします
    response = requests.get(
        ALPHA_VANTAGE_BASE_URL,
        params={"function": function, "symbol": ticker, "apikey": ALPHA_VANTAGE_API_KEY},
    )
    response.raise_for_status()
    return response.json()

In [ ]:
ticker = "IBM"  # "demo" API キーで実データが返るのは IBM のみです。他のティッカーには自分のキーを使ってください
incomeStatement = av_fundamentals("INCOME_STATEMENT", ticker)["annualReports"][0]
balanceSheet = av_fundamentals("BALANCE_SHEET", ticker)["annualReports"][0]
stats = av_fundamentals("OVERVIEW", ticker)

In [ ]:
x1 = ( float(balanceSheet["totalCurrentAssets"]) - float(balanceSheet["totalCurrentLiabilities"]) ) / float(balanceSheet["totalAssets"])

In [ ]:
x2 = float(balanceSheet["retainedEarnings"]) / float(balanceSheet["totalAssets"])

In [ ]:
x3 = float(incomeStatement["ebit"]) / float(balanceSheet["totalAssets"])

In [ ]:
x4 = float(stats["MarketCapitalization"]) / float(balanceSheet["totalLiabilities"])

In [ ]:
6.56 * x1 + 3.26 * x2 + 6.72 * x3 + 1.05 * x4

In [ ]:
def altmanZDoublePrime( ticker ):
    '''
    指定したティッカーについてアルトマンZスコア（ダブルプライム）を計算します。

    ticker = 文字列。Zスコアを計算するティッカーです。Alpha Vantage の
    "demo" API キー（デフォルト）で実データが返るのは "IBM" のみです。他の
    ティッカーを使用するには、環境変数 ALPHA_VANTAGE_API_KEY に自分の無料キー
    （https://www.alphavantage.co/support/#api-key）を設定してください。
    '''
    incomeStatement = av_fundamentals("INCOME_STATEMENT", ticker)["annualReports"][0]
    balanceSheet = av_fundamentals("BALANCE_SHEET", ticker)["annualReports"][0]
    stats = av_fundamentals("OVERVIEW", ticker)
    x1 = ( float(balanceSheet["totalCurrentAssets"]) - float(balanceSheet["totalCurrentLiabilities"]) ) / float(balanceSheet["totalAssets"])
    x2 = float(balanceSheet["retainedEarnings"]) / float(balanceSheet["totalAssets"])
    x3 = float(incomeStatement["ebit"]) / float(balanceSheet["totalAssets"])
    x4 = float(stats["MarketCapitalization"]) / float(balanceSheet["totalLiabilities"])
    return 6.56 * x1 + 3.26 * x2 + 6.72 * x3 + 1.05 * x4

In [ ]:
altmanZDoublePrime("IBM")

In [ ]:
import numpy as np

def altmanZDPImpliedRating( ticker ):
    '''
    企業のアルトマンZスコア（ダブルプライム）からインプライド信用格付けを計算します。
    
    ticker = 文字列。Zスコアを計算する対象を指定するユーザー入力です。大文字と小文字は区別されません。
    '''
    adjZScore = 3.25 + altmanZDoublePrime( ticker )
    zMap = [ 8.15, 7.6, 7.3, 7., 6.85, 6.65, 6.4, 6.25, 5.85, 5.65, 5.25, 4.95, 4.75, 4.5, 4.15, 3.75, 3.2, 2.5, 1.75 ]
    scores = [ "AAA", "AA+", "AA", "AA-", "A+", "A", "A-", "BBB+", "BBB", "BBB-", "BB+", "BB", "BB-", "B+", "B", "B-", "CCC+", "CCC", "CCC-", "D" ] 
    return scores[ zMap.index( np.array( zMap )[ np.array( zMap ) < adjZScore ].max() ) ]

In [ ]:
altmanZDPImpliedRating("IBM")